In [1]:
from __future__ import annotations

# =========================================================
# PS S6E4 - 046a_raw_from_024_family
#
# Purpose:
# - Create a RAW sibling of the 024 / 046 family
# - Keep the family skeleton as much as possible
# - Change input expression from DIGIT-oriented to RAW-oriented
# - Save raw / biased OOF and pred for blend benchmark gating
#
# Positioning:
# - Not a verbatim reproduction of 046a
# - Not 046b itself
# - A controlled family-variant experiment for RAW vs DIGIT comparison
#
# Design choice:
# - Keep frequency remap idea on original categorical columns
# - Keep fold-inside TE using the same custom OrderedTE style
# - Remove digit features entirely
# - Add only light RAW interactions so this does not become a Triko-all-in branch
#
# Intended experiment name suggestion:
#   exp_20260419_050_xgb024_raw_from_family
# =========================================================

In [2]:
import gc
import json
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

In [3]:
# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
class CFG:
    EXP_ID = "exp_20260419_050_xgb024_raw_from_family"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    ID_COL = "id"
    TARGET = "Irrigation_Need"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    SEED = 2026
    N_SPLITS = 5

    N_ESTIMATORS = 3000 # <= 2000
    EARLY_STOPPING_ROUNDS = 100

    # keep bias refine because family comparison should not stop at raw only
    DO_BIAS_REFINE = True
    BIAS_STEPS = [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"
    TAG = "xgb024_raw_from_family"


In [4]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def stable_softmax(logits: np.ndarray) -> np.ndarray:
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def apply_bias_to_proba(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    logits = np.log(np.clip(proba, 1e-15, 1.0)) + bias.reshape(1, -1)
    return stable_softmax(logits)


def greedy_bias_refine(y_true: np.ndarray, base_proba: np.ndarray, steps: list[float]):
    bias = np.zeros(base_proba.shape[1], dtype=np.float64)
    best_score = float(balanced_accuracy_score(y_true, np.argmax(base_proba, axis=1)))

    for step in steps:
        improved = True
        while improved:
            improved = False
            for cls in range(base_proba.shape[1]):
                for sign in (-1.0, 1.0):
                    trial = bias.copy()
                    trial[cls] += sign * step
                    tuned = apply_bias_to_proba(base_proba, trial)
                    score = float(balanced_accuracy_score(y_true, np.argmax(tuned, axis=1)))
                    if score > best_score + 1e-12:
                        bias = trial
                        best_score = score
                        improved = True
    return bias, best_score

In [5]:
# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
seed_everything(CFG.SEED)
ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)

train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP)
y = train[CFG.TARGET].values

drop_cols = [CFG.ID_COL]
train = train.drop(drop_cols, axis=1)
test = test.drop(drop_cols, axis=1)

CATS = [c for c in test.columns if train[c].dtype == object]
NUMS = [c for c in test.columns if c not in CATS]

print("CATS:", CATS)
print("NUMS:", NUMS)
print(train.head())

CATS: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
  Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  \
0     Loamy     4.92          32.58            1.01                     3.05   
1      Clay     7.08          56.61            0.44                     2.00   
2      Clay     5.69          27.71            0.81                     2.83   
3     Sandy     5.65          13.32            1.33                     0.87   
4      Clay     7.96          59.14            0.38                     0.96   

   Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh  \
0          15.01     50.61       725.99            5.90           16.79   
1          22.92     67.86       98

In [6]:
# ---------------------------------------------------------
# OrderedTE same style as 046a reference
# ---------------------------------------------------------
class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train_df, category_cols=None, target_col="target"):
        if category_cols is None:
            category_cols = []

        self.train = train_df.copy()
        self.target_col = target_col
        self.category_cols = category_cols

        self.classes_ = sorted(train_df[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train_df[target_col].value_counts(normalize=True).sort_index().values

        for c in self.category_cols:
            stats_list = []
            for k, cls in enumerate(self.classes_):
                y_binary = (train_df[target_col] == cls).astype(int)

                df = train_df[[c]].copy()
                df["y"] = y_binary.values
                df["cnt"] = 1

                df["cum_cnt"] = df.groupby(c)["cnt"].cumsum() - df["cnt"]
                df["cum_sum"] = df.groupby(c)["y"].cumsum() - df["y"]

                smooth_prior = self.a * self.global_prior_[k]
                te_col = f"{c}_TE_cls{cls}"
                df[te_col] = (df["cum_sum"] + smooth_prior) / (df["cum_cnt"] + self.a)
                df.loc[df["cum_cnt"] == 0, te_col] = self.global_prior_[k]

                self.train[te_col] = df[te_col].values

                stats_df = df.groupby(c)["y"].agg(["count", "sum"]).reset_index()
                stats_df.columns = [c, f"{c}_count_cls{cls}", f"{c}_sum_cls{cls}"]
                stats_df[f"{c}_prior_cls{cls}"] = self.global_prior_[k]
                stats_list.append(stats_df)

            combined_stats = stats_list[0]
            for i in range(1, len(stats_list)):
                combined_stats = combined_stats.merge(stats_list[i], on=c, how="outer")
            setattr(self, f"{c}_stats", combined_stats)

        return self.train

    def transform(self, test_df):
        test_df = test_df.copy()

        for c in self.category_cols:
            stats_df = getattr(self, f"{c}_stats")
            test_df = test_df.merge(stats_df, on=c, how="left")

            for k, cls in enumerate(self.classes_):
                te_col = f"{c}_TE_cls{cls}"
                count_col = f"{c}_count_cls{cls}"
                sum_col = f"{c}_sum_cls{cls}"
                prior_col = f"{c}_prior_cls{cls}"

                if count_col in test_df.columns:
                    test_df[te_col] = (test_df[sum_col] + self.a * test_df[prior_col]) / (test_df[count_col] + self.a)
                    test_df[te_col] = test_df[te_col].fillna(test_df[prior_col])
                    test_df.drop([count_col, sum_col, prior_col], axis=1, inplace=True)
                else:
                    test_df[te_col] = self.global_prior_[k]

        return test_df

In [7]:
# ---------------------------------------------------------
# RAW-from-family feature builder
# ---------------------------------------------------------
def build_features_024_raw_family(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Family-consistent RAW variant.

    Keep:
    - same input split idea as 024/046a
    - frequency-based category remap
    - fold-inside OrderedTE later

    Change:
    - remove all digit features
    - add only light RAW interactions and capped numeric bins
    """
    train_df = train_df.copy()
    test_df = test_df.copy()

    target_col = CFG.TARGET
    id_col = CFG.ID_COL

    if id_col in train_df.columns:
        train_df = train_df.drop(columns=[id_col])
    if id_col in test_df.columns:
        test_df = test_df.drop(columns=[id_col])

    cats = [c for c in test_df.columns if train_df[c].dtype == object]
    nums = [c for c in test_df.columns if c not in cats]

    def add_raw_family_features(df: pd.DataFrame) -> pd.DataFrame:
        d = df.copy()

        # stable category handling
        for c in cats:
            d[c] = d[c].astype(str)

        # light raw interactions
        d["temp_x_humidity"] = d["Temperature_C"] * d["Humidity"]
        d["moisture_x_rain"] = d["Soil_Moisture"] * d["Rainfall_mm"]
        d["sun_x_wind"] = d["Sunlight_Hours"] * d["Wind_Speed_kmh"]
        d["prev_plus_rain"] = d["Previous_Irrigation_mm"] + d["Rainfall_mm"]
        d["prev_minus_moisture"] = d["Previous_Irrigation_mm"] - d["Soil_Moisture"]
        d["field_x_prev"] = d["Field_Area_hectare"] * d["Previous_Irrigation_mm"]
        d["ec_over_oc"] = d["Electrical_Conductivity"] / (np.abs(d["Organic_Carbon"]) + 1e-6)
        d["ph_x_moisture"] = d["Soil_pH"] * d["Soil_Moisture"]

        # minimal threshold/formula-like raw flags
        d["soil_lt_25"] = (d["Soil_Moisture"] < 25).astype("int8")
        d["rain_lt_300"] = (d["Rainfall_mm"] < 300).astype("int8")
        d["temp_gt_30"] = (d["Temperature_C"] > 30).astype("int8")
        d["wind_gt_10"] = (d["Wind_Speed_kmh"] > 10).astype("int8")
        d["deotte_score_light"] = (
            d["soil_lt_25"] * 2
            + d["rain_lt_300"] * 2
            + d["temp_gt_30"]
            + d["wind_gt_10"]
            - (d["Crop_Growth_Stage"].eq("Harvest").astype("int8") * 2)
            - (d["Crop_Growth_Stage"].eq("Sowing").astype("int8") * 2)
            - d["Mulching_Used"].eq("Yes").astype("int8")
        )

        # numeric bins to preserve some discrete view without using digit features
        for c in nums:
            try:
                d[f"{c}_qbin"] = pd.qcut(d[c], q=12, duplicates="drop").astype(str)
            except Exception:
                d[f"{c}_qbin"] = d[c].astype(str)

        # selected pair categories only
        pair_defs = [
            ("Crop_Type", "Crop_Growth_Stage"),
            ("Irrigation_Type", "Water_Source"),
            ("Region", "Season"),
            ("Soil_Type", "Crop_Type"),
        ]
        for a, b in pair_defs:
            d[f"PAIR_{a}_{b}"] = d[a].astype(str) + "__" + d[b].astype(str)

        return d

    train_df = add_raw_family_features(train_df)
    test_df = add_raw_family_features(test_df)

    # constant-column drop based on test side same spirit as 024 safety
    base_test_cols = [c for c in test_df.columns if c != target_col]
    drop_cols = [c for c in base_test_cols if test_df[c].nunique() == 1]
    if drop_cols:
        train_df = train_df.drop(columns=drop_cols, errors="ignore")
        test_df = test_df.drop(columns=drop_cols, errors="ignore")

    raw_qbins = [c for c in test_df.columns if c.endswith("_qbin")]
    pair_cols = [c for c in test_df.columns if c.startswith("PAIR_")]
    category = cats + raw_qbins + pair_cols

    # family-consistent frequency remap
    for c in category:
        freq = train_df[c].value_counts()
        mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
        mapping_default = len(mapping)
        train_df[c] = train_df[c].map(lambda x: mapping.get(x, mapping_default))
        test_df[c] = test_df[c].map(lambda x: mapping.get(x, mapping_default))

    # final numeric columns are every non-category, excluding target
    features = [c for c in train_df.columns if c != target_col]
    X_train_base = train_df[features].copy()
    X_test_base = test_df[features].copy()

    meta = {
        "CATS": cats,
        "NUMS": nums,
        "CATEGORY": category,
        "FEATURES": features,
        "DROP": drop_cols,
        "target_col": target_col,
        "RAW_QBINS": raw_qbins,
        "PAIR_COLS": pair_cols,
    }
    return X_train_base, X_test_base, meta

X_train_base, X_test_base, feature_meta = build_features_024_raw_family(train.copy(), test.copy())
print("CATEGORY:", feature_meta["CATEGORY"])
print("N_FEATURES:", len(feature_meta["FEATURES"]))

CATEGORY: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 'Soil_pH_qbin', 'Soil_Moisture_qbin', 'Organic_Carbon_qbin', 'Electrical_Conductivity_qbin', 'Temperature_C_qbin', 'Humidity_qbin', 'Rainfall_mm_qbin', 'Sunlight_Hours_qbin', 'Wind_Speed_kmh_qbin', 'Field_Area_hectare_qbin', 'Previous_Irrigation_mm_qbin', 'PAIR_Crop_Type_Crop_Growth_Stage', 'PAIR_Irrigation_Type_Water_Source', 'PAIR_Region_Season', 'PAIR_Soil_Type_Crop_Type']
N_FEATURES: 47


In [8]:
# ---------------------------------------------------------
# XGB params
# ---------------------------------------------------------
def make_xgb_params_raw_family():
    # Start near the 046a/046b family zone, but without pretending this is optuna-derived.
    return {
        "n_estimators": CFG.N_ESTIMATORS,
        "max_depth": 3,
        "learning_rate": 0.06,
        "min_child_weight": 3.8,
        "subsample": 0.73,
        "colsample_bytree": 0.60,
        "gamma": 0.52,
        "reg_alpha": 1.5e-4,
        "reg_lambda": 9.89,
        "max_delta_step": 1.60,
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": CFG.SEED,
        "n_jobs": -1,
        "verbosity": 0,
    }

In [9]:
# ---------------------------------------------------------
# CV train
# ---------------------------------------------------------
def train_cv_xgb_raw_family(X_base, y, X_test_base, feature_meta):
    kf = KFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=42)
    oof = np.zeros((len(X_base), 3), dtype=np.float32)
    pred = np.zeros((len(X_test_base), 3), dtype=np.float32)
    fold_scores = []
    best_iterations = []
    fi_rows = []

    CATEGORY = feature_meta["CATEGORY"]
    CATS = feature_meta["CATS"]
    target_col = CFG.TARGET

    params = make_xgb_params_raw_family()

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_base, y), 1):
        X_tr_base = X_base.iloc[tr_idx].copy()
        X_va_base = X_base.iloc[va_idx].copy()
        X_te_base = X_test_base.copy()

        y_tr = pd.Series(y[tr_idx], name=target_col)
        y_va = y[va_idx]

        unique, counts = np.unique(y_tr.values, return_counts=True)
        count_dict = dict(zip(unique, counts))
        avg_count = len(y_tr) / len(unique)
        weights_dict = {cls: avg_count / cnt for cls, cnt in count_dict.items()}
        train_weights = np.array([weights_dict[v] for v in y_tr.values], dtype=np.float32)

        te = OrderedTE(a=1)
        full_df = pd.concat([X_tr_base.reset_index(drop=True), y_tr.reset_index(drop=True)], axis=1)
        full_df["weight"] = train_weights

        te_train = te.fit(
            full_df.sample(frac=1, random_state=42),
            category_cols=CATEGORY,
            target_col=target_col,
        )

        X_tr = te_train.drop([target_col, "weight"], axis=1)
        y_tr_fit = te_train[target_col].values
        train_weights_fit = te_train["weight"].values

        X_va = te.transform(X_va_base.reset_index(drop=True))
        X_te = te.transform(X_te_base.reset_index(drop=True))

        # only original raw categorical columns are dropped, same spirit as 046a
        X_tr = X_tr.drop(columns=CATS, errors="ignore")
        X_va = X_va.drop(columns=CATS, errors="ignore")
        X_te = X_te.drop(columns=CATS, errors="ignore")

        model = xgb.XGBClassifier(**params, early_stopping_rounds=CFG.EARLY_STOPPING_ROUNDS)
        model.fit(
            X_tr,
            y_tr_fit,
            sample_weight=train_weights_fit,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        va_pred = model.predict_proba(X_va)
        te_pred = model.predict_proba(X_te)
        oof[va_idx] = va_pred
        pred += te_pred / CFG.N_SPLITS

        sc = float(balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1)))
        fold_scores.append(sc)
        best_it = int(getattr(model, "best_iteration", -1))
        best_iterations.append(best_it)
        print(f"fold {fold} BA: {sc:.6f} | best_iteration={best_it}")

        booster = model.get_booster()
        gain_dict = booster.get_score(importance_type="gain")
        
        fi = pd.DataFrame({
            "feature": X_tr.columns
        })
        fi[f"gain_f{fold:02d}"] = fi["feature"].map(gain_dict).fillna(0.0)
        
        fi_rows.append(fi)

        del model, X_tr, X_va, X_te, y_tr_fit, train_weights_fit, va_pred, te_pred
        gc.collect()

    raw_cv = float(balanced_accuracy_score(y, np.argmax(oof, axis=1)))
    return raw_cv, fold_scores, best_iterations, oof, pred, fi_rows, params


raw_cv, fold_scores, best_iterations, oof_raw, pred_raw, fi_rows, used_params = train_cv_xgb_raw_family(
    X_train_base, y, X_test_base, feature_meta
)

print("raw_cv:", raw_cv)

fold 1 BA: 0.969799 | best_iteration=2998
fold 2 BA: 0.970043 | best_iteration=2993
fold 3 BA: 0.968500 | best_iteration=2961
fold 4 BA: 0.971728 | best_iteration=2999
fold 5 BA: 0.969050 | best_iteration=2976
raw_cv: 0.9698235647682939


In [10]:
# ---------------------------------------------------------
# Bias refine
# ---------------------------------------------------------
biased_cv = raw_cv
best_bias = np.zeros(3, dtype=np.float64)
oof_biased = oof_raw.copy()
pred_biased = pred_raw.copy()

if CFG.DO_BIAS_REFINE:
    best_bias, biased_cv = greedy_bias_refine(y, oof_raw, CFG.BIAS_STEPS)
    oof_biased = apply_bias_to_proba(oof_raw, best_bias).astype(np.float32)
    pred_biased = apply_bias_to_proba(pred_raw, best_bias).astype(np.float32)
    print("best_bias:", best_bias)
    print("biased_cv:", biased_cv)

best_bias: [-1.01  -1.145  0.   ]
biased_cv: 0.9719717837656457


In [11]:
# ---------------------------------------------------------
# Save artifacts
# ---------------------------------------------------------
np.save(f"{CFG.OUTPUT_DIR}/oof_{CFG.TAG}_proba.npy", oof_raw)
np.save(f"{CFG.OUTPUT_DIR}/pred_{CFG.TAG}_proba.npy", pred_raw)
np.save(f"{CFG.OUTPUT_DIR}/oof_{CFG.TAG}_proba_biased.npy", oof_biased)
np.save(f"{CFG.OUTPUT_DIR}/pred_{CFG.TAG}_proba_biased.npy", pred_biased)
np.save(f"{CFG.OUTPUT_DIR}/best_class_bias.npy", best_bias)

sub_raw = pd.DataFrame({
    CFG.ID_COL: pd.read_csv(CFG.TEST_PATH)[CFG.ID_COL],
    CFG.TARGET: pd.Series(np.argmax(pred_raw, axis=1)).map(CFG.INV_TARGET_MAP),
})
sub_biased = pd.DataFrame({
    CFG.ID_COL: pd.read_csv(CFG.TEST_PATH)[CFG.ID_COL],
    CFG.TARGET: pd.Series(np.argmax(pred_biased, axis=1)).map(CFG.INV_TARGET_MAP),
})
sub_raw.to_csv(f"{CFG.OUTPUT_DIR}/submission_{CFG.TAG}.csv", index=False)
sub_biased.to_csv(f"{CFG.OUTPUT_DIR}/submission_{CFG.TAG}_biased.csv", index=False)
sub_biased.to_csv(f"{CFG.OUTPUT_DIR}/submission.csv", index=False)

pd.DataFrame({"feature": feature_meta["FEATURES"]}).to_csv(
    f"{CFG.OUTPUT_DIR}/feature_columns.csv", index=False
)

# feature importance aggregation
all_gain_cols = []
fi_merged = None
for fold, item in enumerate(fi_rows, start=1):
    # attached dict-like column needs normalization
    col = f"gain_f{fold:02d}"
    if item.empty:
        continue
    mapping = item.set_index("feature")[col].to_dict()
    tmp = pd.DataFrame({"feature": list(mapping.keys()), col: list(mapping.values())})
    fi_merged = tmp if fi_merged is None else fi_merged.merge(tmp, on="feature", how="outer")
    all_gain_cols.append(col)

if fi_merged is None:
    fi_merged = pd.DataFrame({"feature": feature_meta["FEATURES"], "gain_mean": 0.0})
else:
    fi_merged = fi_merged.fillna(0.0)
    fi_merged["gain_mean"] = fi_merged[all_gain_cols].mean(axis=1)
    fi_merged = fi_merged.sort_values("gain_mean", ascending=False)
fi_merged.to_csv(f"{CFG.OUTPUT_DIR}/feature_importance.csv", index=False)

payload = {
    "exp_id": CFG.EXP_ID,
    "base_parent": "024/046 family",
    "variant": "raw_from_family",
    "raw_cv": float(raw_cv),
    "biased_cv": float(biased_cv),
    "best_bias": [float(x) for x in best_bias],
    "fold_scores": [float(x) for x in fold_scores],
    "best_iterations": [int(x) for x in best_iterations],
    "used_params": used_params,
    "notes": {
        "purpose": "raw sibling of 024/046 family for RAW vs DIGIT comparison",
        "digit_features_included": False,
        "ordered_te_in_fold": True,
        "frequency_remap": True,
        "next_gate": "ps-s6e4-safe-blend-v7-two-stage-ridge.ipynb",
    },
}
save_json(payload, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("saved outputs to:", CFG.OUTPUT_DIR)

saved outputs to: /kaggle/working/exp_20260419_050_xgb024_raw_from_family
